In [ ]:
from datetime import datetime
from getpass import getpass
import random

rdm_url = 'staging.rdm.example.com'
idp_name_1 = None  # 'GakuNin RDM IdP'
idp_username_1 = None
idp_password_1 = None
rdm_project_name = 'TEST-WikiImport-{}'.format(datetime.now().strftime('%Y%m%d'))
target_storage_name = 'NII Storage'
target_storage_id = 'osfstorage'
delete_project = True
default_result_path = None
close_on_fail = False
transition_timeout = 60000

In [ ]:
if idp_username_1 is None:
    idp_username_1 = input(prompt=f'Username for {idp_name_1}')
if idp_password_1 is None:
    idp_password_1 = getpass(prompt=f'Password for {idp_username_1}@{idp_name_1}')
(len(idp_username_1), len(idp_password_1))

In [ ]:
import tempfile

work_dir = tempfile.mkdtemp()
if default_result_path is None:
    default_result_path = work_dir
work_dir

# WikiImport

- サブシステム名: アドオン
- ページ/アドオン: WikiImport
- 機能分類: Wiki操作
- シナリオ名: WikiImport
- 用意するテストデータ: URL一覧、アカウント(既存ユーザー1: GRDM)

In [ ]:
import importlib
import pandas as pd

import scripts.playwright
importlib.reload(scripts.playwright)

from scripts.playwright import *
from scripts import grdm

await init_pw_context(close_on_fail=close_on_fail, last_path=default_result_path)

## ウェブブラウザの新規プライベートウィンドウでGRDMトップページを表示する

GRDMトップページが表示されること

In [ ]:
async def _step(page):
    await page.goto(rdm_url)

    # 同意する ボタンが現れるまで待つ
    await expect(page.locator('//button[text() = "同意する"]')).to_be_visible(timeout=transition_timeout)

    # 同意する をクリック
    await page.locator('//button[text() = "同意する"]').click()

    # 同意する が表示されなくなったことを確認
    await expect(page.locator('//button[text() = "同意する"]')).to_have_count(0, timeout=500)

await run_pw(_step)

## ログイン情報を用いてGakuNin RDMにログインする

(IdPに関するログイン情報が与えられた場合、)
GakuNin Embeded DSのプルダウンを展開し、IdPリストから指定されたIdPを選択する。その後、アカウントのID/Passwordを入力して「Login」ボタンを押下する。

(IdPが指定されていない場合、)
CASのログイン操作を実施する。

In [ ]:
async def _step(page):
    await scripts.grdm.login(
        page, idp_name_1, idp_username_1, idp_password_1, transition_timeout=transition_timeout
    )

    await scripts.grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

## プロジェクト一覧に指定されたタイトルのプロジェクトがない場合、指定された名前のプロジェクトを作成する

プロジェクト一覧に当該プロジェクト名が表示されていない場合、「新規プロジェクト作成」をクリックし、その名前を入力、「作成」をクリックする。

In [ ]:
async def _step(page):
    await expect(page.locator('//*[@data-test-create-project-modal-button]')).to_have_count(1)

    await scripts.grdm.ensure_project_exists(page, rdm_project_name, transition_timeout=transition_timeout)

await run_pw(_step)

## 対象のプロジェクトをクリックする

プロジェクトが表示されること

In [ ]:
async def _step(page):
    await page.locator(f'//*[@data-test-dashboard-item-title and text()="{rdm_project_name}"]').click()        

    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await page.locator('//h3[text()="最近の活動"]').click()

await run_pw(_step)

## ファイルタブをクリックする

画面がファイルに切り替わること

In [ ]:
async def _step(page):
    await page.locator('#projectNavFiles a').click()
    time.sleep(1)
    await expect(page.locator('//a[text() = "アドオン"]')).to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_expanded_storage_title_locator(page, target_storage_name)).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

    await expect(page.locator('//h3[text()="最近の活動"]')).not_to_be_visible()

await run_pw(_step)

## 「NII Storage」を選択し、画面上部の「フォルダのアップロード」ボタンをクリックする

フォルダ選択画面が表示される

In [ ]:
async def _step(page):
    await grdm.get_select_storage_title_locator(page, target_storage_name).click()
    await expect(page.locator('//i[contains(@class, "fa-plus")]/../*[text() = "フォルダのアップロード"]')).to_be_enabled()

await run_pw(_step)

## 事前に用意しておいた「WikiImport_1回目」フォルダを選択し「アップロード」ボタンをクリックする

ファイルのアップロード確認メッセージが表示される
> **※本手順の確認内容は自動検証できないため、次手順の中で結果を一括確認します。**

In [ ]:
# ・(補足)自動試験の場合、「確認内容」が確認できないため、手動で作業が必要
async def _step(page):
    pass

await run_pw(_step)

## ファイルのアップロード確認メッセージの「アップロード」ボタンをクリックする

ファイルのアップロードが実施され、「ステータスをアップロード」画面が表示される

In [ ]:
foldername = 'WikiImport_1回目'
folderpath = os.path.join('resources/Datatest-Wiki', foldername)

async def _step(page):
    await grdm.upload_folder(page, folderpath)
    await asyncio.sleep(1)
    await expect(page.locator(f'//*[@role = "progressbar"]')).to_have_count(0, timeout=transition_timeout)
    await expect(page.locator('text="ステータスをアップロード"')).to_be_visible(timeout=transition_timeout)
    
    filecount = sum(len(files) for _, _, files in os.walk(folderpath))
    await expect(page.locator(f'text="{filecount}/{filecount}ファイルが成功しました。"')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "Done"]')).to_be_enabled(timeout=transition_timeout)

await run_pw(_step)

## 「ステータスをアップロード」画面下の「Done」ボタンをクリックする

「ステータスをアップロード」画面が消える

In [ ]:
async def _step(page):
    await page.locator('//a[text() = "Done"]').click()
    await expect(page.locator('text="ステータスをアップロード"')).not_to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_folder_title_locator(page, foldername)).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 手順4〜7を再度実施し、事前に用意しておいた「WikiImport_2回目」〜「WikiImport_5回目」をアップロードする

「NII Storage」の下に「WikiImport_1回目」「WikiImport_2回目」〜「WikiImport_5回目」が表示される

In [ ]:
async def upload_folder_wiki(page, foldername):
    folderpath = os.path.join('resources/Datatest-Wiki', foldername)
    await grdm.upload_folder(page, folderpath)
    await asyncio.sleep(1)
    await expect(page.locator(f'//*[@role = "progressbar"]')).to_have_count(0, timeout=transition_timeout)

    filecount = sum(len(files) for _, _, files in os.walk(folderpath))
    await expect(page.locator('text="ステータスをアップロード"')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator(f'text="{filecount}/{filecount}ファイルが成功しました。"')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//a[text() = "Done"]')).to_be_enabled(timeout=transition_timeout)

    await page.locator('//a[text() = "Done"]').click()
    await expect(page.locator('text="ステータスをアップロード"')).not_to_be_visible(timeout=transition_timeout)
    await expect(grdm.get_select_folder_title_locator(page, foldername)).to_be_visible(timeout=transition_timeout)

async def _step(page):
    await upload_folder_wiki(page, 'WikiImport_2回目')
    await upload_folder_wiki(page, 'WikiImport_3回目')
    await upload_folder_wiki(page, 'WikiImport_4回目')
    await upload_folder_wiki(page, 'WikiImport_5回目')

    await page.reload() 
    await expect(page.locator('//div[@data-index="2"]//span[contains(text(),"WikiImport_1回目")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-index="3"]//span[contains(text(),"WikiImport_2回目")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-index="4"]//span[contains(text(),"WikiImport_3回目")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-index="5"]//span[contains(text(),"WikiImport_4回目")]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//div[@data-index="6"]//span[contains(text(),"WikiImport_5回目")]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## Wikiタブをクリックする

画面がWikiに切り替わること

In [ ]:
async def _step(page):
    await page.get_by_role("link", name="Wiki", exact=True).click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "プロジェクトのWiki"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「インポート」ボタンをクリックする

「Wikiページをインポートする」画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#wikiImport"]').click()
    await expect(page.locator('h3.modal-title', has_text="Wikiページをインポートする")).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## リストボックスで「WikiImport_1回目」を選択し「インポート」ボタンをクリックする

- ボタンが「Wikiページを検証中」→「インポート中」となること
- インポートが完了すると「Import Complete」と表示されること

In [ ]:
async def _step(page):
    await page.select_option('#wikiImportDir', label="WikiImport_1回目")
    await grdm.click_and_expect_alert(page, lambda: page.locator('//button[text()="インポート"]').click(), "Import Complete", transition_timeout=transition_timeout)

await run_pw(_step)

## 「Import Complete」と表示された画面にて「OK」ボタンをクリックする

Wikiの画面に戻り、インポートされたファイル（WikiImport1、WikiImport2）が、画面左の「プロジェクトのWiki」配下に表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport1」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト１（１回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト１（１回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport2」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト２（１回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト２（１回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「インポート」ボタンをクリックする

「Wikiページをインポートする」画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#wikiImport"]').click()
    await expect(page.locator('h3.modal-title', has_text="Wikiページをインポートする")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## リストボックスで「WikiImport_2回目」を選択し「インポート」ボタンをクリックする

ボタンが「Wikiページを検証中」となった後、Wiki名が重複しています と警告画面が表示されること

In [ ]:
async def _step(page):
    await page.select_option('#wikiImportDir', label="WikiImport_2回目")
    await page.locator("#wikiImportSubmit").click()
    await expect(page.locator('h3.modal-title', has_text="Wiki名が重複しています")).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「全てスキップ」を選択し「インポート」ボタンをクリックする

インポートするページがありませんと表示されること

In [ ]:
async def _step(page):
    # ・(補足)自動試験の場合、「確認内容」が確認できないため、手動で作業が必要
    # await page.locator("input#skipAll").check()
    # await grdm.click_and_expect_alert(page, lambda: page.locator("#continueWikiImportSubmit").click(no_wait_after=True), "インポートするページがありません")
    # await expect(page.locator('h3.modal-title', has_text="Wiki名が重複しています")).to_be_visible(timeout=transition_timeout)
    pass

await run_pw(_step)

## 「全て上書き」を選択し「インポート」ボタンをクリックする

- ボタンが「インポート中」となること
- インポートが完了すると「Import Complete」と表示されること

In [ ]:
async def _step(page):
    await page.locator("input#overwriteAll").check()
    await grdm.click_and_expect_alert(page, lambda: page.locator("#continueWikiImportSubmit").click(), "Import Complete", transition_timeout=transition_timeout)

await run_pw(_step)

## 「Import Complete」と表示された画面にて「OK」ボタンをクリックする

Wikiの画面に戻ること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport1」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト１（２回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト１（２回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport2」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト２（２回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト２（２回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 「インポート」ボタンをクリックする

「Wikiページをインポートする」画面が表示されること

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#wikiImport"]').click()
    await expect(page.locator('h3.modal-title', has_text="Wikiページをインポートする")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## リストボックスで「WikiImport_3回目」を選択し「インポート」ボタンをクリックする

ボタンが「Wikiページを検証中」となった後、Wiki名が重複しています と警告画面が表示されること

In [ ]:
async def _step(page):
    await page.select_option('#wikiImportDir', label="WikiImport_3回目")
    await page.locator("#wikiImportSubmit").click()
    await expect(page.locator('h3.modal-title', has_text="Wiki名が重複しています")).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「全て新規に作成」を選択し「インポート」ボタンをクリックする

- ボタンが「インポート中」となること
- インポートが完了すると「Import Complete」と表示されること

In [ ]:
async def _step(page):
    await page.locator("input#createNewAll").check()
    await grdm.click_and_expect_alert(page, lambda: page.locator("#continueWikiImportSubmit").click(), "Import Complete", transition_timeout=transition_timeout)

await run_pw(_step)

## 「Import Complete」と表示された画面にて「OK」ボタンをクリックする

Wikiの画面に戻り、インポートされたファイルが、画面左の「プロジェクトのWiki」配下に「WikiImport1(1)」と「WikiImport2(1)」が表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1(1)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(1)"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport1(1)」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト１（３回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1(1)"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1(1)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト１（３回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport2(1)」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト２（３回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(1)"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(1)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト２（３回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 再度「インポート」ボタンをクリックする

「Wikiページをインポートする」画面が現れること

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#wikiImport"]').click()
    await expect(page.locator('h3.modal-title', has_text="Wikiページをインポートする")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## リストボックスで「WikiImport_4回目」を選択し「インポート」ボタンをクリックする

ボタンが「Wikiページを検証中」となった後、Wiki名が重複しています と警告画面が表示されること

In [ ]:
async def _step(page):
    await page.select_option('#wikiImportDir', label="WikiImport_4回目")
    await page.locator("#wikiImportSubmit").click()
    await expect(page.locator('h3.modal-title', has_text="Wiki名が重複しています")).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「全て新規に作成」を選択し「インポート」ボタンをクリックする

- ボタンが「インポート中」となること
- インポートが完了すると「Import　Complete」と表示されること

In [ ]:
async def _step(page):
    await page.locator("input#createNewAll").check()
    await grdm.click_and_expect_alert(page, lambda: page.locator("#continueWikiImportSubmit").click(), "Import Complete", transition_timeout=transition_timeout)

await run_pw(_step)

## 「Import　Complete」と表示された画面にて「OK」ボタンをクリックする

Wikiの画面に戻り、インポートされたファイルが、画面左の「プロジェクトのWiki」配下に「WikiImport1(2)」と「WikiImport2(2)」が表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1(2)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(2)"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport1(2)」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト１（４回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1(2)"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1(2)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト１（４回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport2(2)」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト２（４回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(2)"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(2)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト２（４回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 再度「インポート」ボタンをクリックする

「Wikiページをインポートする」画面が現れること

In [ ]:
async def _step(page):
    await page.locator('div[data-target="#wikiImport"]').click()
    await expect(page.locator('h3.modal-title', has_text="Wikiページをインポートする")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## リストボックスで「WikiImport_5回目」を選択し「インポート」ボタンをクリックする

ボタンが「Wikiページを検証中」となった後、Wiki名が重複しています と警告画面が表示されること

In [ ]:
async def _step(page):
    await page.select_option('#wikiImportDir', label="WikiImport_5回目")
    await page.locator("#wikiImportSubmit").click()
    await expect(page.locator('h3.modal-title', has_text="Wiki名が重複しています")).to_be_visible(timeout=transition_timeout)
    time.sleep(1)

await run_pw(_step)

## 「ページ毎に選択」ボタンをクリックする

ページを選択する画面が表示されること

In [ ]:
async def _step(page):
    await page.locator("#perFileDefinition").click()
    await expect(page.locator('select[name="WikiImportOperationPerSelect"]').first).to_be_visible()

await run_pw(_step)

## 全て「スキップ」を選択し「インポート」ボタンをクリックする

インポートするページがありませんと表示されること

In [ ]:
async def _step(page):
    # ・(補足)自動試験の場合、「確認内容」が確認できないため、手動で作業が必要
    # # WikiImport1 → スキップ
    # wiki1 = page.locator('li#WikiImport1')
    # await wiki1.locator('select[name="WikiImportOperationPerSelect"]').select_option("skip")

    # # WikiImport2 → スキップ
    # wiki2 = page.locator('li#WikiImport2')
    # await wiki2.locator('select[name="WikiImportOperationPerSelect"]').select_option("skip")
    # await grdm.click_and_expect_alert(page, lambda: page.locator("#continueWikiImportSubmit").click(), "インポートするページがありません")
    pass

await run_pw(_step)

## 「WikiImport1」を上書きにする「WikiImport2」を新規に作成にする「インポート」ボタンをクリックする

- ボタンが「インポート中」となること
- インポートが完了すると「Import Complete」と表示されること

In [ ]:
async def _step(page):
    # WikiImport1 → 上書き
    wiki1 = page.locator('li#WikiImport1')
    await wiki1.locator('select[name="WikiImportOperationPerSelect"]').select_option("overwrite")

    # WikiImport2 → 新規に作成
    wiki2 = page.locator('li#WikiImport2')
    await wiki2.locator('select[name="WikiImportOperationPerSelect"]').select_option("createNew")
    await grdm.click_and_expect_alert(page, lambda: page.locator("#continueWikiImportSubmit").click(), "Import Complete", transition_timeout=transition_timeout)

await run_pw(_step)

## 「Import　Complete」と表示された画面にて「OK」ボタンをクリックする

Wikiの画面に戻り、インポートされたファイルが、画面左の「プロジェクトのWiki」配下に「WikiImport2(3)」が表示されること

In [ ]:
async def _step(page):
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(3)"]')).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport1」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト１（５回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport1"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト１（５回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## 画面左の「プロジェクトのWiki」内にある「WikiImport2(3)」をクリックする

画面右のWiki表示領域にWikiの内容が表示され、1行目に「Wikiインポートテスト２（５回目）」と表示されること

In [ ]:
async def _step(page):
    await page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(3)"]').click()
    await expect(page.locator('//*[contains(@class, "title-text")]//*[text() = "WikiImport2(3)"]')).to_be_visible(timeout=transition_timeout)
    await expect(page.locator("#wikiViewRender h1", has_text="Wikiインポートテスト２（５回目）")).to_be_visible(timeout=transition_timeout)

await run_pw(_step)

## (指定がある場合) プロジェクトを削除する

In [ ]:
async def _step(page):
    if not delete_project:
        return
    await scripts.grdm.delete_project(page)
    await scripts.grdm.expect_dashboard(page, transition_timeout=transition_timeout)

await run_pw(_step)

終了処理を実施。

In [ ]:
await finish_pw_context()

In [ ]:
!rm -fr {work_dir}